In [0]:
from pyspark.sql import functions as F

CATALOG = "dbr_dev"
# dbutils.widgets.get("catalog")

GOLD = f"{CATALOG}.skybound_gold"

CHECKPOINT_PATH = f"/Volumes/{CATALOG}/skybound_bronze/checkpoints/weather_to_gold_snapshot"

In [0]:
%sql
SELECT * FROM dbr_dev.skybound_gold.agg_current_airport_weather

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

def refresh_current_weather_snapshot():

    # Latest observation per station from the fact table.

    latest_obs = spark.sql(f"""
        SELECT *
        FROM (
            SELECT
                f.*,
                ROW_NUMBER() OVER (
                    PARTITION BY f.station_id
                    ORDER BY f.obs_time DESC
                ) AS rn
            FROM {GOLD}.fact_airport_weather_reports f
        )
        WHERE rn = 1
    """)

    snapshot = (
        latest_obs
        .select(
            latest_obs.station_id.alias("airport_icao"),
            latest_obs.airport_name,
            latest_obs.airport_country,
            latest_obs.country_code,
            latest_obs.airport_latitude,
            latest_obs.airport_longitude,
            latest_obs.airport_elevation,

            latest_obs.obs_time,
            latest_obs.temp_c,
            latest_obs.dewpoint_c,
            latest_obs.wind_dir_degrees,
            latest_obs.wind_dir_variable,
            latest_obs.wind_speed_kt,
            latest_obs.wind_gust_kt,
            latest_obs.visibility_sm,
            latest_obs.visibility_unlimited,
            latest_obs.altim_hpa,
            latest_obs.cover_description,
            latest_obs.cloud_base_ft,
            latest_obs.weather_description,
            latest_obs.flight_category,
            latest_obs.flight_category_name,
            latest_obs.category_severity,
            latest_obs.is_ifr,
            latest_obs.has_precipitation,
            latest_obs.has_freezing,
            latest_obs.has_thunderstorm,
            latest_obs.has_fog,
            latest_obs.metar_type,
            F.current_timestamp().alias("_snapshot_updated_at")
        )
    )

    target = DeltaTable.forName(spark, f"{GOLD}.agg_current_airport_weather")
    (
        target.alias("t")
        .merge(snapshot.alias("s"), "t.airport_icao = s.airport_icao")
        .whenMatchedUpdate(
            condition="s.obs_time > t.obs_time",
            set={col: f"s.{col}" for col in snapshot.columns}
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

refresh_current_weather_snapshot()